# Diagnosis: Where could reranking help?

Find labeled questions with Recall@30 greater than Recall@5 using the rebuilt Qwen retrieval. Counts and example ranks are observed anew.

**Saved execution:** the outputs below preserve the recorded Qwen run. Opening this notebook does not require rerunning its cells. For deliberate replay, set `NOTEBOOK_MODE=replay` before starting the kernel. Execute mode can prepare missing work when paid calls are explicitly enabled through `NOTEBOOK_ALLOW_MODEL_CALLS=1` and `OPENROUTER_API_KEY`; judging has additional action controls. This editorial cleanup did not execute any cells or change model settings.

**Study scope:** the frozen larger corpus contains **152,576 passages** associated with **15,585 fully covered question IDs** from MS MARCO v2.1 validation; this is a subset, not the full validation split. Experiment 1 uses a separate **10,210-passage corpus**, 31 seeded baseline questions, the original saved effervescent diagnostic, and a separately saved IMEI diagnostic. Experiment 2 samples **500 opportunities + 200 controls**. Experiment 3 contains **79 sampled pairs, including IMEI**. Experiment 4 times **8 question pairs / 16 answers**. IMEI is excluded from the 31-question baseline averages but is already included in the frozen Experiment 2 and Experiment 3 samples; displaying it again adds no observation.

**Workflow:** Experiment 1 (Baseline) --> **this notebook Diagnosis** --> Experiment 2 (Targeted Reranking) --> Experiment 3 (Answer Quality) --> Experiment 4 (Latency Tradeoff)


## 1. Load the embedding-only records

**What:** load saved questions, their 30 embedding-ranked candidates and annotated support IDs.

**Why:** make the opportunity concrete and inspectable without repeating embedding calls.

Open this notebook from its extracted folder, or set `DIAGNOSTIC_ROOT` to that folder. All calculations below use embedding ranks. No reranker ordering, answer text, SQuAD or RAGAS score determines membership.

In [1]:
from pathlib import Path
import json
import os
import numpy as np
import pandas as pd
from IPython.display import display

diagnostic_folder = "diagnosis-retrieval-opportunity"
notebook_name = "Diagnosis-Retrieval-Opportunity.ipynb"
if os.environ.get("DIAGNOSTIC_ROOT"):
    ROOT = Path(os.environ["DIAGNOSTIC_ROOT"]).expanduser().resolve()
    if not (ROOT / notebook_name).is_file() and not (ROOT / "data").is_dir():
        raise FileNotFoundError("DIAGNOSTIC_ROOT must point to the diagnostic notebook folder.")
else:
    ROOT = None
    for parent in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        matches = [candidate for candidate in [parent, parent / diagnostic_folder]
                   if (candidate / notebook_name).is_file()]
        if not matches:
            matches = [path.parent for path in parent.glob(f"*/{diagnostic_folder}/{notebook_name}")]
        matches = sorted({candidate.resolve() for candidate in matches})
        if len(matches) > 1:
            raise ValueError("Multiple diagnostic notebooks found; set DIAGNOSTIC_ROOT to the intended folder.")
        if matches:
            ROOT = matches[0]
            break
    if ROOT is None:
        raise FileNotFoundError("Cannot locate the diagnostic notebook; set DIAGNOSTIC_ROOT to its folder.")
DATA = ROOT / "data"
required_data = ["opportunity_records.csv", "embedding_candidates.csv.gz", "annotated_support.csv",
                 "example_support_excerpt.csv", "provenance.json", "reported_full_validation_summary.json"]
if not all((DATA / name).is_file() for name in required_data):
    import sys
    PROJECT = next((p for p in [ROOT, *ROOT.parents]
                    if (p / "configs/execution.json").is_file()), None)
    if PROJECT is None:
        raise FileNotFoundError("Fresh diagnostic preparation needs the full repository and requirements-runtime.txt.")
    sys.path.insert(0, str(PROJECT))
    from shared_scoring.execution import ensure_diagnostic_data
    from shared_scoring.notebook_runtime import NotebookSession, StudyNotReady
    session = NotebookSession(PROJECT)
    MODE = session.mode
    cached_data = session.settings['runtime_directory'] / 'diagnosis' / session.retrieval_directory().name / 'data'
    if all((cached_data / name).is_file() for name in required_data):
        DATA = cached_data
    elif MODE == 'execute':
        session.preflight('diagnosis')
        # First execution: query embedding and retrieval; no reranker/generator calls.
        DATA = ensure_diagnostic_data(ROOT, PROJECT)
    else:
        raise StudyNotReady('Qwen embedding-only diagnosis has not been produced yet.')

provenance = json.loads((DATA / "provenance.json").read_text())
reported = json.loads((DATA / "reported_full_validation_summary.json").read_text())
records = pd.read_csv(DATA / "opportunity_records.csv", dtype={"query_id": str})
candidates = pd.read_csv(DATA / "embedding_candidates.csv.gz", dtype={"query_id": str})
supports = pd.read_csv(DATA / "annotated_support.csv", dtype={"query_id": str})
TOP_CONTEXT, TOP_CANDIDATES = 5, 30
TOLERANCE = 1e-12

assert records.query_id.is_unique
support_ids_by_query = supports.groupby("query_id").passage_id.agg(set)
for row in records.itertuples():
    assert set(json.loads(row.known_supporting_passage_ids)) == support_ids_by_query[row.query_id]
assert not candidates.duplicated(["query_id", "passage_id"]).any()
assert not supports.duplicated(["query_id", "passage_id"]).any()
assert set(candidates.query_id) == set(records.query_id) == set(supports.query_id)
rank_check = candidates.groupby("query_id").embedding_rank.agg(list)
assert rank_check.map(lambda values: sorted(values) == list(range(1, 31))).all()

settings = provenance["retrieval_settings"]
display(pd.DataFrame([
    ("Dataset", "MS MARCO v2.1 validation"),
    ("Embedding model", settings["embedding_model"]),
    ("Embedding dimensions", settings["embedding_dimensions"]),
    ("Retrieval", "FAISS IndexIVFFlat; L2-normalized cosine search"),
    ("Candidate / context budgets", "30 / 5 passages"),
    ("Passage limit", "1,800 characters in the upstream retrieval setup"),
    ("Records available here", len(records)),
    ("Model calls", "Fresh retrieval: inspect runtime request logs" if provenance.get("fresh_run") else 0),
], columns=["Setting", "Value"]))


Validation retrieval: 512/15,585


Validation retrieval: 1,024/15,585


Validation retrieval: 1,536/15,585


Validation retrieval: 2,048/15,585


Validation retrieval: 2,560/15,585


...

Validation retrieval: 14,336/15,585


Validation retrieval: 14,848/15,585


Validation retrieval: 15,360/15,585


Validation retrieval: 15,585/15,585


,Setting,Value
0,Dataset,MS MARCO v2.1 validation
1,Embedding model,qwen/qwen3-embedding-4b
2,Embedding dimensions,2560
3,Retrieval,FAISS IndexIVFFlat; L2-normalized cosine search
4,Candidate / context budgets,30 / 5 passages
5,Passage limit,"1,800 characters in the upstream retrieval setup"
6,Records available here,1085
7,Model calls,Fresh retrieval: inspect runtime request logs


## 2. Diagnose the frozen validation subset

Compare annotated-support coverage at five versus thirty passages across the **15,585-question frozen subset**. Of these, **10,242** have known support labels. The saved upstream summary reports Recall@5 **0.8918**, Recall@30 **0.9910**, and **1,085 opportunity questions**.

Recall@k is the fraction of a question's annotated supporting passages found in its first k retrieved passages. These are known-label measurements, not exhaustive relevance judgments. The table below loads the saved upstream summary; it does not estimate population averages from opportunity rows.

**Preserved-output label:** the table's “full-validation” heading refers here to this frozen subset, not the full MS MARCO validation split.


In [2]:
display(pd.DataFrame([
    ("Validation questions", reported["validation_questions"]),
    ("Questions with support annotations", reported["annotated_questions"]),
    ("Reported mean Recall@5", reported["mean_recall_at_5"]),
    ("Reported mean Recall@30", reported["mean_recall_at_30"]),
    ("Reported mean recall gap", reported["mean_recall_gap"]),
    ("All annotated support in top five", reported["all_support_top5_questions"]),
    ("All annotated support in top thirty", reported["all_support_top30_questions"]),
    ("Additional support in ranks 6–30", reported["opportunity_questions"]),
    ("Some support absent from top thirty", reported["some_support_absent_top30_questions"]),
], columns=["Previously reported full-validation result", "Value"])
    .style.format({"Value": "{:.4f}"})
    .format("{:,.0f}", subset=pd.IndexSlice[[0, 1, 5, 6, 7, 8], "Value"]))

,Previously reported full-validation result,Value
0,Validation questions,"15,585"
1,Questions with support annotations,"10,242"
2,Reported mean Recall@5,0.8918
3,Reported mean Recall@30,0.9910
4,Reported mean recall gap,0.0992
5,All annotated support in top five,"9,062"
6,All annotated support in top thirty,"10,139"
7,Additional support in ranks 6–30,"1,085"
8,Some support absent from top thirty,103


## 3. Define the opportunity before testing reranking

**What:** select labeled questions where **Recall@30 > Recall@5**.

**Why:** some annotated evidence is already retrieved at ranks 6–30, but would be left out of a five-passage context. Reordering could help; it has not yet been shown to help.

| Embedding-only situation | What it tells us |
|---|---|
| Recall@30 > Recall@5 | Evidence below the cutoff is recoverable by selection or reordering. |
| Recall@30 = Recall@5 = 1 | All annotated support is already in the top five. |
| Recall@30 = Recall@5 < 1 | Missing annotated support is outside the top thirty; reordering cannot recover those missing IDs. |

Query IDs are stable identifiers, not spreadsheet row numbers. `query_row` is the original dataset row position. The rule depends on support annotations and embedding ranks, not on a generated answer.

In [3]:
records["recall_gap"] = records.recall_at_30 - records.recall_at_5
assert records.has_known_support_labels.eq(True).all()
opportunity = records.loc[records.recall_gap.gt(TOLERANCE)].copy()
assert len(opportunity) == len(records) == reported["opportunity_questions"]
assert np.allclose(opportunity.recall_gap, opportunity.recall_30_minus_recall_5,
                   rtol=0, atol=TOLERANCE)

display(pd.DataFrame([
    ("All validation questions", reported["validation_questions"]),
    ("Questions with annotations", reported["annotated_questions"]),
], columns=["Denominator", "Questions"]).assign(
    opportunity_questions=len(opportunity),
    opportunity_share_pct=lambda table: 100 * len(opportunity) / table.Questions,
).round(2))


,Denominator,Questions,opportunity_questions,opportunity_share_pct
0,All validation questions,15585,1085,6.96
1,Questions with annotations,10242,1085,10.59


## 4. Show the actual opportunity questions

**What:** inspect questions with no annotated support in the top five, and questions missing some complementary support.

**Why:** explain what the selection rule means for real queries rather than stop at an aggregate.

These are embedding-only categories. They are not the later gain/unchanged/loss groups. The displayed examples are ordered deterministically by original dataset row; they are illustrations, not representative random samples. The complete selected question table is `data/opportunity_records.csv`.

In [4]:
opportunity["embedding_situation"] = np.where(
    opportunity.recall_at_5.le(TOLERANCE),
    "No annotated support in top five",
    "Some support in top five; additional support below cutoff",
)
display(opportunity.groupby("embedding_situation").agg(
    questions=("query_id", "size"),
    mean_recall_at_5=("recall_at_5", "mean"),
    mean_recall_at_30=("recall_at_30", "mean"),
    mean_recall_gap=("recall_gap", "mean"),
).round(4))

view_columns = ["query_row", "query_id", "question", "recall_at_5", "recall_at_30", "recall_gap"]
for situation, group in opportunity.groupby("embedding_situation"):
    print("\n" + situation)
    display(group.sort_values("query_row")[view_columns].head(5))

,questions,mean_recall_at_5,mean_recall_at_30,mean_recall_gap
embedding_situation,,,,
No annotated support in top five,957,0.0000,0.9969,0.9969
Some support in top five; additional support below cutoff,128,0.5109,0.9954,0.4845



No annotated support in top five


,query_row,query_id,question,recall_at_5,recall_at_30,recall_gap
2,38,1090278,symptoms of air pollution exposure,0.0,1.0,1.0
3,54,174249,does xpress bet charge to deposit money in you...,0.0,1.0,1.0
4,73,180613,english is the official state language in some...,0.0,1.0,1.0
5,84,330883,how much would graduate marketers make?,0.0,1.0,1.0
6,86,79604,can you set google chrome as your default web ...,0.0,1.0,1.0



Some support in top five; additional support below cutoff


,query_row,query_id,question,recall_at_5,recall_at_30,recall_gap
0,1,1102431,why did rachel carson write an obligation to e...,0.500000,1.0,0.500000
1,26,114414,"current weather in volcano, ca",0.666667,1.0,0.333333
11,136,236184,how is alkali made,0.500000,1.0,0.500000
14,179,190280,foods to firm up stool,0.500000,1.0,0.500000
15,181,178627,effects of detox juice cleanse,0.500000,1.0,0.500000


## 5. Verify coverage directly from passage IDs

**What:** join annotated support IDs to the embedding-ranked candidate IDs, then count support in the first five and thirty.

**Why:** verify the recall columns from evidence membership rather than trust copied scores.

The denominator includes **all** annotated support for each question. A support passage outside the top thirty has a missing candidate rank and counts as a miss. Finding one of two annotated passages gives recall 0.5. This audit verifies the bundled selected records, not all 15,585 questions in the frozen subset.


In [5]:
support_positions = supports[["query_id", "passage_id"]].merge(
    candidates, on=["query_id", "passage_id"], how="left", validate="one_to_one"
)
support_positions["hit_5"] = support_positions.embedding_rank.le(TOP_CONTEXT)
support_positions["hit_30"] = support_positions.embedding_rank.notna()
support_positions["below_cutoff"] = support_positions.embedding_rank.between(6, 30)

coverage = support_positions.groupby("query_id").agg(
    annotated_passages=("passage_id", "size"),
    support_in_top5=("hit_5", "sum"),
    support_in_top30=("hit_30", "sum"),
    recoverable_support=("below_cutoff", "sum"),
    recomputed_recall_at_5=("hit_5", "mean"),
    recomputed_recall_at_30=("hit_30", "mean"),
    first_support_rank=("embedding_rank", "min"),
).reset_index()
verified = opportunity.merge(coverage, on="query_id", validate="one_to_one")
for k in [5, 30]:
    assert np.allclose(verified[f"recall_at_{k}"], verified[f"recomputed_recall_at_{k}"],
                       rtol=0, atol=TOLERANCE)
assert verified.recoverable_support.ge(1).all()
print(f"Verified support-ID coverage for all {len(verified):,} opportunity questions.")
display(verified.sort_values("query_row")[[
    "query_id", "question", "annotated_passages", "support_in_top5",
    "support_in_top30", "recoverable_support", "first_support_rank",
]].head(10))

Verified support-ID coverage for all 1,085 opportunity questions.


,query_id,question,annotated_passages,support_in_top5,support_in_top30,recoverable_support,first_support_rank
0,1102431,why did rachel carson write an obligation to e...,2,1,2,1,1.0
1,114414,"current weather in volcano, ca",3,2,3,1,4.0
2,1090278,symptoms of air pollution exposure,1,0,1,1,7.0
3,174249,does xpress bet charge to deposit money in you...,1,0,1,1,7.0
4,180613,english is the official state language in some...,1,0,1,1,15.0
5,330883,how much would graduate marketers make?,1,0,1,1,12.0
6,79604,can you set google chrome as your default web ...,1,0,1,1,6.0
7,1101279,do physicians pay for insurance from their sal...,1,0,1,1,6.0
8,96252,complex variable functions,1,0,1,1,6.0
9,264835,how long is the flight from dfw to punta cana,1,0,1,1,7.0


## 6. Inspect one question’s cutoff

Inspect an observed opportunity in the new Qwen retrieval. Its annotated support ranks are recomputed.


In [6]:
EXAMPLE_ID = "325543"  # Change this to inspect another ID in opportunity_records.csv.
if provenance.get("fresh_run") and EXAMPLE_ID not in set(verified.query_id):
    EXAMPLE_ID = provenance["inspect_query_id"]
question = verified.set_index("query_id").loc[EXAMPLE_ID]
print(f"Query {EXAMPLE_ID}: {question['question']}")
display(pd.DataFrame([
    ("Annotated-support Recall@5", question.recomputed_recall_at_5),
    ("Annotated-support Recall@30", question.recomputed_recall_at_30),
    ("First annotated-support rank", question.first_support_rank),
], columns=["Embedding-only measurement", "Value"]))

support_ids = set(supports.loc[supports.query_id.eq(EXAMPLE_ID), "passage_id"])
example_candidates = candidates.loc[candidates.query_id.eq(EXAMPLE_ID)].copy()
example_candidates["annotated_support"] = example_candidates.passage_id.isin(support_ids)
example_candidates["would_enter_top5_context"] = example_candidates.embedding_rank.le(5)
display(example_candidates.sort_values("embedding_rank").loc[
    lambda table: table.would_enter_top5_context | table.annotated_support
])
excerpts = pd.read_csv(DATA / "example_support_excerpt.csv", dtype={"query_id": str})
for row in excerpts.loc[excerpts.query_id.eq(EXAMPLE_ID)].itertuples():
    actual_rank = example_candidates.set_index("passage_id").loc[row.passage_id, "embedding_rank"]
    assert actual_rank == row.embedding_rank
    excerpt_origin = "Fresh saved" if provenance.get("fresh_run") else "Archived"
    print(f"\n{excerpt_origin} annotated-support excerpt: {row.passage_id}, embedding rank {actual_rank}")
    print(row.text)
if not excerpts.query_id.eq(EXAMPLE_ID).any():
    print("No passage excerpt bundled for this query; its IDs and ranks are still inspectable.")

Query 1102431: why did rachel carson write an obligation to endure


,Embedding-only measurement,Value
0,Annotated-support Recall@5,0.5
1,Annotated-support Recall@30,1.0
2,First annotated-support rank,1.0


,query_id,passage_id,embedding_rank,annotated_support,would_enter_top5_context
0,1102431,p_a7d7c1627719a2739347,1,True,True
1,1102431,p_798501edc35d7af8b254,2,False,True
2,1102431,p_4b697b2575e680939e1e,3,False,True
3,1102431,p_aca5d9930a137042d66a,4,False,True
4,1102431,p_9291f65e1a1abad7633a,5,False,True
8,1102431,p_df702fcc14ec0419b842,9,True,False



Fresh saved annotated-support excerpt: p_a7d7c1627719a2739347, embedding rank 1
The Obligation to Endure by Rachel Carson Rachel Carson's essay on The Obligation to Endure, is a very convincing argument about the harmful uses of chemicals, pesticides, herbicides, and fertilizers on the environment.

Fresh saved annotated-support excerpt: p_df702fcc14ec0419b842, embedding rank 9
Carson believes that as man tries to eliminate unwanted insects and weeds, however he is actually causing more problems by polluting the environment with, for example, DDT and harming living things. Carson adds that the intensification of agriculture is causing other major problems, like newly developed or created insects and diseases.


## 7. Hand a fixed sample to targeted reranking

The diagnosis identifies **1,085** opportunity questions using Recall@30 > Recall@5. Experiment 2 samples **500** opportunities and **200** no-gap controls with the configured seed before reranking. It reuses each selected question's exact thirty candidates. The separate effervescent diagnostic does not enter those group averages.


## Optional appendix: reproduce selection from the upstream subset table

Supply the original upstream CSV with `query_id`, `has_known_support_labels`, `recall_at_5` and `recall_at_30`. An optional `question` column makes the displayed records easier to inspect. Set `FULL_VALIDATION_METRICS_PATH` privately, then enable the switch.

This is an optional local check, not needed to replay the bundled opportunity records. It verifies the frozen-subset count and exact opportunity-ID set. It does not sample controls or request a model. The provenance file retains the original upstream file hash for comparison.


In [7]:
VERIFY_FULL_VALIDATION = False
if VERIFY_FULL_VALIDATION:
    import hashlib
    full_path = Path(os.environ["FULL_VALIDATION_METRICS_PATH"]).expanduser().resolve()
    if full_path.suffix.lower() != ".csv":
        raise ValueError("Export the upstream per-question metrics to CSV for this minimal tutorial.")
    full = pd.read_csv(full_path, dtype={"query_id": str})
    required = {"query_id", "has_known_support_labels", "recall_at_5", "recall_at_30"}
    if not required.issubset(full.columns):
        raise ValueError(f"Missing columns: {sorted(required - set(full.columns))}")
    assert full.query_id.is_unique and len(full) == reported["validation_questions"]
    labels = full.has_known_support_labels.astype(str).str.lower().map({"true": True, "false": False})
    assert labels.notna().all(), "Support-label flags must be True/False."
    assert int(labels.sum()) == reported["annotated_questions"]
    full_gap = full.recall_at_30 - full.recall_at_5
    selected = full.loc[labels & full_gap.gt(TOLERANCE)].copy()
    assert set(selected.query_id) == set(opportunity.query_id)
    print(f"Full-validation selection matches all {len(selected):,} bundled opportunity IDs.")
    actual_hash = hashlib.sha256(full_path.read_bytes()).hexdigest()
    print("Exact original file bytes:", actual_hash == provenance["upstream_per_query_metrics_sha256"])
    display(selected.head(10))
else:
    print("Full-validation check disabled: only the bundled opportunity records were recalculated.")

Full-validation check disabled: only the bundled opportunity records were recalculated.


### Reproducibility

The runtime data directory contains question-level recall, ranked candidate IDs, support annotations, provenance, and the saved upstream subset summary. Some filenames and original output headings retain “full_validation” for compatibility. Their scope is the frozen 15,585-question subset.
